# Baseline TF-IDF + Regressão Logística: treino e validação (Issue 7)

Este notebook reproduz dois experimentos e salva os resultados em `results/`:

1. **Baseline oficial:** treina com todo o `train.csv` e avalia uma única vez no `test.csv` oficial.
2. **Validação interna:** separa 20% do `train.csv` de forma estratificada por nota; ajusta TF-IDF e LR nos 80% restantes e avalia nos 20% reservados. O `test.csv` não participa desse experimento.

A validação bruta, com todas as colunas originais, é salva em `data/raw/validation.csv` e pode ser baixada. O arquivo não é versionado porque a licença dos reviews não está especificada. O default usa todos os dados (`SAMPLE_N=0`); um número positivo serve para um smoke run.

In [ ]:
# Célula 1 — repositório e dependências
import pathlib

if pathlib.Path('/content').exists() and not pathlib.Path('Projeto-IA-Replica-RatingPrediction').exists():
    !git clone https://github.com/Mur1love/Projeto-IA-Replica-RatingPrediction.git
    %cd Projeto-IA-Replica-RatingPrediction

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Preserva numpy/pandas que já vêm com o runtime do Colab.
    !pip install -q scikit-learn matplotlib nltk joblib
else:
    !pip install -q -r requirements.txt

In [ ]:
# Célula 2 — upload de train.csv e test.csv
import os
from pathlib import Path

DATA_DIR = Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = Path(os.environ.get('TRAIN_CSV', str(DATA_DIR / 'train.csv')))
TEST_CSV = Path(os.environ.get('TEST_CSV', str(DATA_DIR / 'test.csv')))

try:
    from google.colab import files
    print('Faça upload dos dois CSVs oficiais: train.csv e test.csv')
    uploaded = files.upload()
    for filename, contents in uploaded.items():
        target = DATA_DIR / Path(filename).name
        target.write_bytes(contents)
        print(f'{target}: {len(contents) / 1e6:.1f} MB')
    TRAIN_CSV = DATA_DIR / 'train.csv'
    TEST_CSV = DATA_DIR / 'test.csv'
except ImportError:
    print('Runtime local: usando os caminhos TRAIN_CSV e TEST_CSV configurados acima.')

assert TRAIN_CSV.is_file(), f'Não encontrei {TRAIN_CSV}; envie train.csv.'
assert TEST_CSV.is_file(), f'Não encontrei {TEST_CSV}; envie test.csv.'
print(f'Treino: {TRAIN_CSV} | Teste oficial: {TEST_CSV}')

In [ ]:
# Célula 3 — imports, fallback para branch ainda não publicada e configuração
import hashlib
import json
import os
import pathlib
import sys

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'src' / 'preprocessing').is_dir():
        ROOT = candidate
        sys.path.insert(0, str(ROOT))
        break
assert ROOT is not None, 'Não encontrei a raiz do repositório.'

import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from src.preprocessing.normalization import normalize, normalize_dataframe
from src.preprocessing.encoding import encode_rating
from src.preprocessing.embedding import (
    DEFAULT_PARAMS,
    build_vectorizer,
    fit_vectorizer,
    transform_texts,
)

# O notebook também funciona se o Colab clonou main antes do merge da Issue 7.
try:
    from src.data.splitting import split_train_validation
except ModuleNotFoundError as exc:
    if exc.name != 'src.data.splitting':
        raise

    def split_train_validation(
        df, *, validation_size=0.2, random_state=42,
        text_col='text', rating_col='rating'
    ):
        keep = df[text_col].map(normalize).ne('')
        eligible = df.loc[keep].reset_index(drop=True)
        train_part, validation_part = train_test_split(
            eligible, test_size=validation_size,
            stratify=eligible[rating_col], random_state=random_state
        )
        return train_part.reset_index(drop=True), validation_part.reset_index(drop=True)

try:
    from src.models.baseline import train_logistic_regression
except ModuleNotFoundError as exc:
    if exc.name != 'src.models.baseline':
        raise
    from sklearn.linear_model import LogisticRegression

    def train_logistic_regression(
        X_train, y_train, *, C=1.0, max_iter=1000, random_state=42
    ):
        model = LogisticRegression(
            C=C, max_iter=max_iter, solver='lbfgs', random_state=random_state
        )
        return model.fit(X_train, y_train)

try:
    from src.evaluation.metrics import compute_metrics
except ModuleNotFoundError as exc:
    if exc.name != 'src.evaluation.metrics':
        raise
    from sklearn.metrics import (
        accuracy_score, f1_score, mean_absolute_error, mean_squared_error
    )

    def compute_metrics(y_true_labels, y_pred_labels):
        y_true = np.asarray(y_true_labels, dtype=np.int64)
        y_pred = np.asarray(y_pred_labels, dtype=np.int64)
        if y_true.shape != y_pred.shape:
            raise ValueError('y_true e y_pred devem ter o mesmo tamanho')
        rating_true, rating_pred = y_true + 1, y_pred + 1
        return {
            'accuracy': float(accuracy_score(y_true, y_pred)),
            'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
            'mae': float(mean_absolute_error(rating_true, rating_pred)),
            'rmse': float(np.sqrt(mean_squared_error(rating_true, rating_pred))),
        }

try:
    from src.evaluation.plots import save_confusion_matrix
except ModuleNotFoundError as exc:
    if exc.name != 'src.evaluation.plots':
        raise
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

    def save_confusion_matrix(
        y_true, y_pred, path, *, display_labels=(1, 2, 3, 4, 5)
    ):
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(cm, display_labels=list(display_labels)).plot(
            ax=ax, cmap='Blues', colorbar=False
        )
        ax.set_title('Matriz de confusão — Regressão Logística (TF-IDF)')
        ax.set_xlabel('Predito (rating)')
        ax.set_ylabel('Verdadeiro (rating)')
        path = pathlib.Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        return path

SEED = 42
VALIDATION_SIZE = 0.20
C = 1.0
MAX_ITER = 1000
SAMPLE_N = int(os.environ.get('SAMPLE_N', '0'))
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', 'results'))
VALIDATION_OUT = Path(
    os.environ.get('VALIDATION_OUT', str(DATA_DIR / 'validation.csv'))
)
METRICS_DIR = OUTPUT_DIR / 'metrics'
FIGURES_DIR = OUTPUT_DIR / 'figures'
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('scikit-learn:', sklearn.__version__)
print('seed:', SEED, '| validation_size:', VALIDATION_SIZE, '| SAMPLE_N:', SAMPLE_N)
print('TF-IDF:', DEFAULT_PARAMS)

In [ ]:
# Célula 4 — baseline original: treino completo → teste oficial
from IPython.display import Image, display


def stratified_sample(df, n, seed):
    if n <= 0 or n >= len(df):
        return df.copy()
    sample, _ = train_test_split(
        df, train_size=n, stratify=df['rating'], random_state=seed
    )
    return sample.reset_index(drop=True)

raw_train_source = pd.read_csv(TRAIN_CSV)
raw_test_source = pd.read_csv(TEST_CSV)
raw_train_full = raw_train_source
raw_test_full = raw_test_source
if SAMPLE_N > 0:
    raw_train_full = stratified_sample(raw_train_source, SAMPLE_N, SEED)
    n_test_sample = min(len(raw_test_source), max(200, SAMPLE_N // 4))
    raw_test_full = stratified_sample(raw_test_source, n_test_sample, SEED)

train_full = normalize_dataframe(
    raw_train_full[['text', 'rating']], 'text', empty='drop'
)
test_full = normalize_dataframe(
    raw_test_full[['text', 'rating']], 'text', empty='drop'
)
vectorizer_full, X_train_full = fit_vectorizer(
    train_full['text'], build_vectorizer(**DEFAULT_PARAMS)
)
X_test_full = transform_texts(vectorizer_full, test_full['text'])
y_train_full = encode_rating(train_full['rating']).to_numpy()
y_test_full = encode_rating(test_full['rating']).to_numpy()
model_full = train_logistic_regression(
    X_train_full, y_train_full, C=C, max_iter=MAX_ITER, random_state=SEED
)
predictions_test = model_full.predict(X_test_full)
metrics_full = compute_metrics(y_test_full, predictions_test)

full_figure = FIGURES_DIR / 'confusion_matrix_logreg.png'
save_confusion_matrix(y_test_full, predictions_test, full_figure)
full_metrics = {
    'seed': SEED,
    'lr_params': {'C': C, 'max_iter': MAX_ITER, 'solver': 'lbfgs'},
    'vectorizer_params': dict(DEFAULT_PARAMS),
    'normalization': {},
    'n_train': int(len(train_full)),
    'n_test': int(len(test_full)),
    'n_dropped_train': int(len(raw_train_full) - len(train_full)),
    'n_dropped_test': int(len(raw_test_full) - len(test_full)),
    'sklearn_version': sklearn.__version__,
    'metrics_logreg': metrics_full,
}
full_metrics_path = METRICS_DIR / 'baseline_logreg.json'
full_metrics_path.write_text(
    json.dumps(full_metrics, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
)
print('Baseline completo:', metrics_full)
print(f"Treino/teste: {len(train_full)} / {len(test_full)}")
display(Image(filename=str(full_figure)))

In [ ]:
# Célula 5 — split 80/20, treino e avaliação de validação (sem usar o test)
train_raw, validation_raw = split_train_validation(
    raw_train_full, validation_size=VALIDATION_SIZE,
    random_state=SEED, text_col='text', rating_col='rating'
)
VALIDATION_OUT.parent.mkdir(parents=True, exist_ok=True)
validation_raw.to_csv(VALIDATION_OUT, index=False)

train_validation = normalize_dataframe(
    train_raw[['text', 'rating']], 'text', empty='drop'
)
validation = normalize_dataframe(
    validation_raw[['text', 'rating']], 'text', empty='drop'
)
vectorizer_validation, X_train_validation = fit_vectorizer(
    train_validation['text'], build_vectorizer(**DEFAULT_PARAMS)
)
X_validation = transform_texts(vectorizer_validation, validation['text'])
y_train_validation = encode_rating(train_validation['rating']).to_numpy()
y_validation = encode_rating(validation['rating']).to_numpy()
model_validation = train_logistic_regression(
    X_train_validation, y_train_validation,
    C=C, max_iter=MAX_ITER, random_state=SEED
)
predictions_validation = model_validation.predict(X_validation)
metrics_validation = compute_metrics(y_validation, predictions_validation)

validation_figure = FIGURES_DIR / 'confusion_matrix_logreg_validation.png'
save_confusion_matrix(y_validation, predictions_validation, validation_figure)
validation_metrics = {
    'seed': SEED,
    'validation_size': VALIDATION_SIZE,
    'n_source_rows': int(len(raw_train_full)),
    'n_dropped_empty': int(len(raw_train_full) - len(train_raw) - len(validation_raw)),
    'n_train': int(len(train_validation)),
    'n_validation': int(len(validation)),
    'validation_csv': str(VALIDATION_OUT),
    'validation_csv_sha256': hashlib.sha256(VALIDATION_OUT.read_bytes()).hexdigest(),
    'sklearn_version': sklearn.__version__,
    'lr_params': {'C': C, 'max_iter': MAX_ITER, 'solver': 'lbfgs', 'random_state': SEED},
    'vectorizer_params': dict(DEFAULT_PARAMS),
    'metrics_validation': metrics_validation,
}
validation_metrics_path = METRICS_DIR / 'baseline_logreg_validation.json'
validation_metrics_path.write_text(
    json.dumps(validation_metrics, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8'
)

comparison_rows = []
for name in ('accuracy', 'f1_macro', 'mae', 'rmse'):
    comparison_rows.append({
        'metric': name,
        'full_train_official_test': metrics_full[name],
        'train_80_validation': metrics_validation[name],
        'validation_minus_test': metrics_validation[name] - metrics_full[name],
        'better_direction': 'higher' if name in ('accuracy', 'f1_macro') else 'lower',
    })
comparison_path = METRICS_DIR / 'baseline_logreg_comparison.csv'
pd.DataFrame(comparison_rows).to_csv(comparison_path, index=False)
print(f"Treino/validação: {len(train_validation)} / {len(validation)}")
print('Métricas da validação:', metrics_validation)
display(pd.DataFrame(comparison_rows))
display(Image(filename=str(validation_figure)))

In [ ]:
# Célula 6 — baixar split local e os artefatos dos dois experimentos
try:
    from google.colab import files
    for artifact in [
        VALIDATION_OUT,
        full_metrics_path,
        full_figure,
        validation_metrics_path,
        validation_figure,
        comparison_path,
    ]:
        files.download(str(artifact))
except ImportError:
    print('Runtime local: resultados disponíveis em', OUTPUT_DIR)
    print('Split local:', VALIDATION_OUT)